In [4]:
import cv2
import pytesseract
import pandas as pd
from pathlib import Path
import re

In [5]:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

processed_dir = Path("../data/processed")
df = pd.read_csv("../data/annotated/ground_truth.csv")
ground_truth = pd.read_csv("../data/annotated/ground_truth.csv")

In [6]:
def extract_amount(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return None
    
    lines = text.split('\n')
    for i, line in enumerate(lines):
        if re.search(r'(?:total amount due|total|amount due|amt due|grand total|sum|tota)', line, re.IGNORECASE):
            for check_line in lines[i:i+2]:
                match = re.search(r'(\d{1,3}(?:,\d{3})*\.\d{2})', check_line)
                if match:
                    try:
                        val = float(match.group(1).replace(',', ''))
                        if val >= 5.0 and 1.0 <= val <= 99999.0:
                            return round(val, 2)
                    except:
                        continue
    
    blocklist = ['vat', 'vatable', 'subtotal', 'cash', 'change', 'discount', 'zero rated', 'exempt']
    candidates = []
    for match in re.finditer(r'\d{1,3}(?:,\d{3})*\.\d{2}', text):
        num_str = match.group(0)
        start = match.start()
        preceding = text[max(0, start-50):start].lower()
        if any(word in preceding for word in blocklist):
            continue
        try:
            val = float(num_str.replace(',', ''))
            if 1.0 <= val <= 99999.0:
                candidates.append(val)
        except:
            continue
    
    return round(max(candidates), 2) if candidates else None

In [7]:
psm_modes = [3, 4, 6, 11, 12]
all_results = {}
all_texts = {}

for psm in psm_modes:
    print(f"Running PSM {psm}...")
    psm_results = []
    psm_texts = []
    
    for _, row in df.iterrows():
        img_path = processed_dir / row['filename']
        img = cv2.imread(str(img_path))
        
        if img is None:
            psm_results.append({
                "filename": row["filename"],
                "condition": row["condition"],
                "extracted_amount": None
            })
            psm_texts.append("")
            continue
        
        config = f"--oem 1 --psm {psm}"
        extracted_text = pytesseract.image_to_string(img, config=config).strip()
        extracted_amount = extract_amount(extracted_text)
        
        psm_results.append({
            "filename": row["filename"],
            "condition": row["condition"],
            "extracted_amount": extracted_amount
        })
        psm_texts.append(extracted_text)
    
    all_results[psm] = psm_results
    all_texts[psm] = psm_texts
    print(f"✅ PSM {psm} done\n")

Running PSM 3...
✅ PSM 3 done

Running PSM 4...
✅ PSM 4 done

Running PSM 6...
✅ PSM 6 done

Running PSM 11...
✅ PSM 11 done

Running PSM 12...
✅ PSM 12 done



In [8]:
final = ground_truth.copy()

for psm in psm_modes:
    psm_df = pd.DataFrame(all_results[psm])
    final[f'psm{psm}_amount'] = psm_df['extracted_amount'].values
    final[f'psm{psm}_correct'] = abs(
        final['amount'] - final[f'psm{psm}_amount'].fillna(-9999)
    ) < 2.0

In [9]:
print("=" * 50)
print("        PSM COMPARISON SUMMARY")
print("=" * 50)
print(f"{'PSM':<5} | {'Extracted':<10} | {'Correct':<8} | {'Accuracy'}")
print("-" * 50)

best_accuracy = 0
best_psm_acc = 0
best_extraction = 0
best_psm_ext = 0

for psm in psm_modes:
    extracted = final[f'psm{psm}_amount'].notna().sum()
    correct = final[f'psm{psm}_correct'].sum()
    accuracy = correct / len(final) * 100
    marker = " ← BEST" if psm == 12 else ""
    print(f"{psm:<5} | {extracted}/{len(final):<8} | {correct}/{len(final):<6} | {accuracy:.1f}%{marker}")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_psm_acc = psm
    if extracted > best_extraction:
        best_extraction = extracted
        best_psm_ext = psm

print("=" * 50)
print(f"\nHighest Accuracy  : PSM {best_psm_acc} ({best_accuracy:.1f}%)")
print(f"Highest Extraction: PSM {best_psm_ext} ({best_extraction}/{len(final)})")
print(f"\n✅ Selected PSM 12 for final output (highest accuracy: {best_accuracy:.1f}%)")

        PSM COMPARISON SUMMARY
PSM   | Extracted  | Correct  | Accuracy
--------------------------------------------------
3     | 18/22       | 15/22     | 68.2%
4     | 18/22       | 14/22     | 63.6%
6     | 19/22       | 15/22     | 68.2%
11    | 19/22       | 18/22     | 81.8%
12    | 20/22       | 19/22     | 86.4% ← BEST

Highest Accuracy  : PSM 12 (86.4%)
Highest Extraction: PSM 12 (20/22)

✅ Selected PSM 12 for final output (highest accuracy: 86.4%)


In [10]:
print("\nSaving PSM 12 results as final OCR output...\n")

final_results = []

for i, row in df.iterrows():
    img_path = processed_dir / row['filename']
    img = cv2.imread(str(img_path))

    extracted_text = all_texts[12][i]

    if img is not None:
        config = "--oem 1 --psm 12"
        data = pytesseract.image_to_data(
            img, config=config,
            output_type=pytesseract.Output.DATAFRAME
        )
        valid_data = data[data['conf'] > 0]
        avg_conf = valid_data['conf'].mean() / 100.0 if not valid_data.empty else 0.0
    else:
        avg_conf = 0.0

    line_count = len([l for l in extracted_text.split('\n') if l.strip()])

    final_results.append({
        "filename": row["filename"],
        "condition": row["condition"],
        "extracted_text": extracted_text,
        "avg_confidence": round(avg_conf, 4)
    })

    print(f"✅ {row['filename']:18} | Conf: {avg_conf:.3f} | Lines: {line_count}")

ocr_df = pd.DataFrame(final_results)
ocr_df.to_csv("../outputs/metrics/easyocr_results.csv", index=False)

print("\n" + "=" * 70)
print("✅ PSM 12 results saved to easyocr_results.csv")
print("✅ Notebooks 04 and 05 will use PSM 12 output")
print(ocr_df[['filename', 'avg_confidence']].to_string(index=False))


Saving PSM 12 results as final OCR output...

✅ receipt_01.jpg     | Conf: 0.718 | Lines: 57
✅ receipt_02.jpg     | Conf: 0.713 | Lines: 75
✅ receipt_03.jpg     | Conf: 0.772 | Lines: 67
✅ receipt_04.jpg     | Conf: 0.719 | Lines: 58
✅ receipt_05.jpg     | Conf: 0.763 | Lines: 30
✅ receipt_06.jpg     | Conf: 0.621 | Lines: 88
✅ receipt_07.jpg     | Conf: 0.610 | Lines: 58
✅ receipt_08.jpg     | Conf: 0.630 | Lines: 63
✅ receipt_09.jpg     | Conf: 0.536 | Lines: 97
✅ receipt_10.jpg     | Conf: 0.603 | Lines: 77
✅ receipt_11.jpg     | Conf: 0.762 | Lines: 91
✅ receipt_12.jpg     | Conf: 0.623 | Lines: 67
✅ receipt_13.jpg     | Conf: 0.561 | Lines: 92
✅ receipt_14.jpg     | Conf: 0.566 | Lines: 81
✅ receipt_15.jpg     | Conf: 0.631 | Lines: 100
✅ receipt_16.jpg     | Conf: 0.633 | Lines: 94
✅ receipt_17.jpg     | Conf: 0.657 | Lines: 104
✅ receipt_18.jpg     | Conf: 0.663 | Lines: 111
✅ receipt_19.jpg     | Conf: 0.627 | Lines: 107
✅ receipt_20.jpg     | Conf: 0.644 | Lines: 62
✅ receipt